In [2]:
import pandas as pd
import duckdb

df = pd.read_csv("Data/spotify-2000-joined-clean.csv")

df.head()

,title,artist,genre,year,bpm,energy,danceability,loudness_db,liveness,valence,duration,acousticness,speechiness,popularity,decade,popularity_segment,winner
0,sunrise,norah_jones,adult_standards,2004,157,30,53,-14,11,68,201,94,3,71,2000,media-alta,True
1,the_pretender,foo_fighters,alternative_metal,2007,173,96,43,-4,3,37,269,0,4,76,2000,alta,True
2,without_me,eminem,detroit_hip_hop,2002,112,67,91,-3,24,66,290,0,7,82,2000,alta,True
3,uninvited,alanis_morissette,alternative_rock,2005,127,54,38,-5,9,19,276,2,3,57,2000,media-baja,True
4,cry_me_a_river,justin_timberlake,dance_pop,2002,74,65,62,-7,10,56,288,57,18,74,2000,alta,True


In [3]:
con = duckdb.connect("spotify.duckdb")

In [4]:
con.execute("""
CREATE OR REPLACE TABLE spotify AS
SELECT *
FROM df
""")

In [6]:
con.execute("""
SELECT *
FROM spotify
LIMIT 5
""").fetchdf()

,title,artist,genre,year,bpm,energy,danceability,loudness_db,liveness,valence,duration,acousticness,speechiness,popularity,decade,popularity_segment,winner
0,sunrise,norah_jones,adult_standards,2004,157,30,53,-14,11,68,201,94,3,71,2000,media-alta,True
1,the_pretender,foo_fighters,alternative_metal,2007,173,96,43,-4,3,37,269,0,4,76,2000,alta,True
2,without_me,eminem,detroit_hip_hop,2002,112,67,91,-3,24,66,290,0,7,82,2000,alta,True
3,uninvited,alanis_morissette,alternative_rock,2005,127,54,38,-5,9,19,276,2,3,57,2000,media-baja,True
4,cry_me_a_river,justin_timberlake,dance_pop,2002,74,65,62,-7,10,56,288,57,18,74,2000,alta,True


In [8]:
# Segmentación del catálogo por década
# ¿Qué décadas presentan mejor desempeño promedio para ayudar a la selección del contenido musical?
query_decade = con.execute("""
SELECT 
    decade,
    COUNT(*) AS total_songs,
    ROUND(AVG(popularity),2) AS avg_popularity

FROM spotify

GROUP BY decade

ORDER BY decade

""").fetchdf()

query_decade

,decade,total_songs,avg_popularity
0,1950,9,66.22
1,1960,158,64.16
2,1970,353,62.13
3,1980,344,59.45
4,1990,331,59.47
5,2000,400,57.98
6,2010,399,56.90


Permite identificar décadas con mayores niveles de popularidad promedio para apoyar decisiones relacionadas con selección y organización de playlists

In [9]:
# Segmentación del catálogo por nivel de popularidad
# ¿Cómo se distrbuyen las canciones según su desempeño relativo?

query_popularity = con.execute("""

SELECT

    popularity_segment,
    COUNT(*) AS total_songs,
    ROUND(AVG(energy),2) AS avg_energy,
    ROUND(AVG(danceability),2) AS avg_danceability

FROM spotify

GROUP BY popularity_segment

ORDER BY popularity_segment

""").fetchdf()

query_popularity

,popularity_segment,total_songs,avg_energy,avg_danceability
0,alta,458,63.28,56.69
1,baja,499,57.02,51.28
2,media-alta,497,59.90,53.56
3,media-baja,540,58.89,51.83


In [10]:
# ¿Las canciones más populares presentan diferencias en su nivel emocional y capacidad de ser bailables?

query_features = con.execute("""

SELECT 

    popularity_segment,

    ROUND(AVG(danceability),2) AS avg_danceability,
    ROUND(AVG(valence),2) AS avg_valence,
    ROUND(AVG(liveness),2) AS avg_liveness

FROM spotify

GROUP BY popularity_segment

ORDER BY

CASE
    WHEN popularity_segment='Baja' THEN 1
    WHEN popularity_segment='Media-Baja' THEN 2
    WHEN popularity_segment='Media-Alta' THEN 3
    WHEN popularity_segment='Alta' THEN 4

END

""").fetchdf()

query_features

,popularity_segment,avg_danceability,avg_valence,avg_liveness
0,media-alta,53.56,51.68,18.39
1,alta,56.69,52.22,16.97
2,media-baja,51.83,47.03,18.73
3,baja,51.28,47.14,21.81


Las canciones con mayor popularidad parecen mostrar niveles superiores de Danceability y Valence, mientras que Liveness disminuye. Esto puede sugerir que canciones más bailables y con emociones positivas tienen mejor aceptación dentro del catálogo analizado.

Si el objetivo es optimizar playlists premium, podrían priorizarse canciones con características similares a las observadas en segmentos de alta popularidad.

In [11]:
# ¿Cómo han cambiado las caracterísitcas musicales a través del tiempo?

query_trends = con.execute("""

SELECT

    decade,

    ROUND(AVG(danceability),2) AS avg_danceability,
    ROUND(AVG(valence),2) AS avg_valence,
    ROUND(AVG(energy),2) AS avg_energy

FROM spotify

GROUP BY decade

ORDER BY decade

""").fetchdf()

query_trends

,decade,avg_danceability,avg_valence,avg_energy
0,1950,51.78,72.00,45.33
1,1960,48.20,52.99,49.44
2,1970,51.58,54.31,56.23
3,1980,57.02,57.14,61.03
4,1990,51.47,42.91,59.86
5,2000,53.41,46.23,64.36
6,2010,54.77,45.06,61.11


Permite identificar si los gustos o estilos musicales muestran cambios a través de generaciones y si ciertas características se vuelven más frecuentes.